# Challenge 2: Semi-Supervised Learning para Series de Tiempo

**Objetivo:** El propósito de este notebook es construir y evaluar un modelo de Semi-Supervised Learning (SSL) para una tarea de clasificación de series de tiempo. Usaremos datos de consumo eléctrico de la EIA (U.S. Energy Information Administration) para predecir si el consumo de un estado en un mes particular será "alto" en comparación con su historial.

**Metodología:**
1.  **Descarga y Preprocesamiento de Datos:** Obtendremos los datos de ventas de electricidad, los limpiaremos y crearemos features de series de tiempo (rezagos, promedios móviles, etc.).
2.  **Definición del Target:** Crearemos una variable objetivo binaria que indique si el consumo (`sales`) de un mes está en el 30% superior del historial de ese estado.
3.  **Modelos Baseline:** Entrenaremos dos modelos base (Regresión Logística y Random Forest) utilizando solo una pequeña fracción de datos etiquetados (10%).
4.  **Semi-Supervised Learning (Self-Training):** Implementaremos un clasificador Self-Training que utiliza el modelo base para generar "pseudo-etiquetas" en los datos no etiquetados y se re-entrena iterativamente.
5.  **Evaluación:** Compararemos el rendimiento del modelo SSL contra los baselines en un conjunto de prueba (held-out) para determinar si el uso de datos no etiquetados mejoró el rendimiento.

---
### 1. Importación de Librerías

En esta sección, importamos todas las librerías necesarias para el análisis, incluyendo `pandas` para manipulación de datos, `scikit-learn` para el modelado y `requests` para la descarga de datos.

In [1]:
import os
import time
import logging
import numpy as np
import pandas as pd
import requests
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

### 2. Configuración y Constantes

Aquí definimos las constantes que se utilizarán a lo largo del notebook, como la clave de la API para la EIA, el nombre del archivo de caché, las semillas para la reproducibilidad de los experimentos y las métricas que evaluaremos. También configuramos el sistema de logging para registrar el progreso y los resultados.

In [2]:
API_KEY      = os.environ.get("EIA_API_KEY", "KrDmNDoo6L83uPEXqgizPLnFPmDUwWbqGge4mz10")
CACHE_FILE   = "eia_retail_sales.csv"
SEEDS        = [42, 123, 777]
METRICS_KEYS = ["f1_macro", "auc", "f1_0", "f1_1", "prec_1", "rec_1"]

os.makedirs("logs",    exist_ok=True)
os.makedirs("results", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("logs/challenge2.log", mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)



### 3. Descarga de Datos

Esta función se encarga de descargar los datos de ventas minoristas de electricidad de la API de la EIA. Para evitar descargas repetidas y acelerar la ejecución, los datos se guardan en un archivo CSV local (`eia_retail_sales.csv`) que funciona como caché. Si el archivo ya existe, los datos se cargan desde allí.

In [3]:
def fetch_eia_retail_sales(api_key: str) -> pd.DataFrame:
    """
    Descarga EIA Electricity Retail Sales mensual por estado × sector.
    Incluye: sales (MWh), price (¢/kWh), revenue ($M), customers (#).
    Paginación automática — guarda caché local para no repetir descarga.
    """
    if os.path.exists(CACHE_FILE):
        log.info(f"Cargando datos desde caché: {CACHE_FILE}")
        df = pd.read_csv(CACHE_FILE)
        log.info(f"  {len(df):,} registros cargados")
        return df

    log.info("Descargando EIA Retail Sales (puede tomar ~2 min)...")
    base_url   = "https://api.eia.gov/v2/electricity/retail-sales/data/"
    all_rows, offset, page = [], 0, 5000

    params = {
        "frequency"         : "monthly",
        "data[0]"           : "sales",
        "data[1]"           : "price",
        "data[2]"           : "revenue",
        "data[3]"           : "customers",
        "start"             : "1990-01",
        "end"               : "2026-04",
        "sort[0][column]"   : "period",
        "sort[0][direction]": "asc",
        "length"            : page,
        "api_key"           : api_key,
    }

    while True:
        params["offset"] = offset
        r = requests.get(base_url, params=params, timeout=30)
        r.raise_for_status()
        body    = r.json()
        records = body.get("response", {}).get("data", [])
        if not records:
            break
        all_rows.extend(records)
        total = int(body.get("response", {}).get("total", 0))
        log.info(f"  descargado: {len(all_rows):,} / {total:,}")
        if len(all_rows) >= total:
            break
        offset += page
        time.sleep(0.25)

    df = pd.DataFrame(all_rows)
    df.to_csv(CACHE_FILE, index=False)
    log.info(f"Guardado en {CACHE_FILE}  ({len(df):,} filas)")
    return df



### 4. Preprocesamiento y Creación de Features

Esta es la etapa más importante del análisis. La función `preprocess` realiza las siguientes tareas:

1.  **Limpieza de Datos:** Convierte las columnas a tipos de datos numéricos y de fecha.
2.  **Creación del Target:** Define la variable objetivo `target`. Un registro tiene `target = 1` si sus ventas (`sales`) son mayores o iguales al percentil 70 del historial de ventas de ese estado. Este cálculo se hace de forma cuidadosa para no usar información del futuro (evitar *look-ahead bias*).
3.  **Creación de Features:** Genera un conjunto de características (features) basadas en el historial de la serie de tiempo. Estas incluyen:
    *   **Lags:** Valores de ventas de meses anteriores.
    *   **Estadísticas Móviles:** Promedios y desviación estándar sobre ventanas de tiempo.
    *   **Momentum:** Diferencia de ventas entre periodos.
    *   **Z-score:** Normalización de las ventas recientes.
    *   **Features de Calendario:** Estacionalidad capturada con funciones seno y coseno.
    *   **Tendencia:** Una variable que captura el paso del tiempo.

El resultado son dos arreglos de NumPy: `X` (features) y `y` (target).

In [4]:
def preprocess(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """
    TARGET: ¿alto consumo eléctrico para ESTE estado en ESTE mes?
    ──────────────────────────────────────────────────────────────
    Se calcula el percentil 70 de 'sales' DENTRO de cada estado
    (usando solo datos hasta t-1 para evitar look-ahead).
    target = 1  si  sales(t) >= percentil_70_histórico_del_estado
    target = 0  en  caso contrario

    Por qué top 30% y no top 20%:
      - Con datos de 35 años (~420 meses/estado) el top 20% = 84 meses
        por entidad, suficiente estadísticamente.
      - Top 30% da ~30/70 de balance, más manejable para SSL con pocos
        datos etiquetados que el 20/80 original.

    FEATURES (todas sin leakage — solo datos del PASADO):
      lag1, lag2, lag3, lag6  : ventas de meses anteriores
      roll3/6/12_mean         : media móvil sobre valores pasados
      roll6_std               : volatilidad reciente
      mom_1m, mom_3m          : momentum de ventas
      z_score_12m             : posición relativa dentro del año reciente
      price_lag1              : precio del mes anterior
      month_sin/cos (×2)      : estacionalidad Fourier
      year_trend              : índice temporal
      state_enc               : estado codificado (para RF)
    """
    df = df.copy()

    for col in ["sales", "price", "revenue", "customers"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["period"] = pd.to_datetime(df["period"])
    df = (df.sort_values(["stateid", "period"])
            .reset_index(drop=True))
    df = df.dropna(subset=["sales"]).reset_index(drop=True)

    log.info(f"Registros iniciales: {len(df):,} | "
             f"Estados: {df['stateid'].nunique()} | "
             f"Rango: {df['period'].min().date()} → {df['period'].max().date()}")

    # ── TARGET: top 30% dentro de cada estado ─────────────────────────
    # expanding() sobre shift(1) → el umbral de cada mes se calcula
    # SOLO con el historial previo de ese estado (sin ver el futuro).
    grp_sales = df.groupby("stateid")["sales"]

    expanding_q70 = grp_sales.transform(
        lambda x: x.shift(1).expanding(min_periods=24).quantile(0.70)
    )
    df["target"] = (df["sales"] >= expanding_q70).astype(int)

    # Filas con umbral no calculable (primeros 24 meses por estado) → drop
    df = df.dropna(subset=["sales"]).copy()
    df = df[expanding_q70.notna()].reset_index(drop=True)

    log.info(f"Target (top 30% dentro del estado): "
             f"clase 1 = {df['target'].mean():.1%} | "
             f"clase 0 = {1 - df['target'].mean():.1%}")

    # ── FEATURES ─────────────────────────────────────────────────────
    grp = df.groupby("stateid")

    # Lags de ventas (solo pasado)
    df["lag1"]  = grp["sales"].shift(1)
    df["lag2"]  = grp["sales"].shift(2)
    df["lag3"]  = grp["sales"].shift(3)
    df["lag6"]  = grp["sales"].shift(6)
    df["lag12"] = grp["sales"].shift(12)

    # Estadísticas móviles (shift(1) interno para no ver el mes actual)
    _past = grp["sales"].transform(lambda x: x.shift(1))
    df["roll3_mean"]  = _past.rolling(3,  min_periods=2).mean()
    df["roll6_mean"]  = _past.rolling(6,  min_periods=3).mean()
    df["roll12_mean"] = _past.rolling(12, min_periods=6).mean()
    df["roll6_std"]   = _past.rolling(6,  min_periods=3).std()

    # Momentum
    df["mom_1m"] = df["lag1"] - df["lag2"]
    df["mom_3m"] = df["lag1"] - df["lag3"]

    # Z-score dentro del año reciente: ¿lag1 está alto o bajo respecto
    # a los últimos 12 meses de ese estado?
    df["z_score_12m"] = (
        (df["lag1"] - df["roll12_mean"])
        / (df["roll6_std"].replace(0, np.nan) + 1e-9)
    )

    # Precio del mes anterior (feature de contexto económico)
    df["price_lag1"] = grp["price"].shift(1)

    # Estacionalidad Fourier (captura patrones invierno/verano mejor que
    # usar el número de mes directamente)
    df["month_sin"]  = np.sin(2 * np.pi * df["period"].dt.month / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df["period"].dt.month / 12)
    df["month_sin2"] = np.sin(4 * np.pi * df["period"].dt.month / 12)
    df["month_cos2"] = np.cos(4 * np.pi * df["period"].dt.month / 12)

    # Tendencia temporal
    df["year_trend"] = (
        (df["period"].dt.year - 1990) * 12 + df["period"].dt.month
    )

    # Estado codificado
    df["state_enc"] = df["stateid"].astype("category").cat.codes

    df = df.dropna().reset_index(drop=True)

    FEATURES = [
        "lag1", "lag2", "lag3", "lag6", "lag12",
        "roll3_mean", "roll6_mean", "roll12_mean", "roll6_std",
        "mom_1m", "mom_3m",
        "z_score_12m",
        "price_lag1",
        "month_sin", "month_cos", "month_sin2", "month_cos2",
        "year_trend",
        "state_enc",
    ]

    X = df[FEATURES].values
    y = df["target"].values

    log.info(f"Dataset final: {X.shape[0]:,} filas × {X.shape[1]} features")
    log.info(f"Clases: 0 = {(y==0).sum():,} | 1 = {(y==1).sum():,}")
    return X, y

 

### 5. División de Datos (Train/Test y Labeled/Unlabeled)

Para simular un escenario de Semi-Supervised Learning, dividimos los datos de la siguiente manera:

1.  **Conjunto de Prueba (Test Set):** Se reserva un 20% de los datos como un conjunto de prueba final. Este conjunto **nunca** se utiliza durante el entrenamiento, ni siquiera en el proceso SSL.
2.  **Conjunto de Entrenamiento (Pool):** El 80% restante se divide en:
    *   **Datos Etiquetados (Labeled):** Un 10% del pool, que usaremos para entrenar los modelos supervisados.
    *   **Datos no Etiquetados (Unlabeled):** El 90% restante, que el modelo SSL usará para aprender.

La división se hace de forma estratificada para mantener la misma proporción de clases en cada conjunto.

In [5]:
def get_splits(X, y, seed: int,
               labeled_frac: float = 0.10,
               test_frac:    float = 0.20):
    """
    Paso 1 → 20% held-out test  (estratificado, BLOQUEADO para SSL)
    Paso 2 → Del 80% restante:
               10% labeled   (entrenamiento supervisado)
               90% unlabeled (pool para SSL)
    """
    sss_test = StratifiedShuffleSplit(
        n_splits=1, test_size=test_frac, random_state=seed)
    idx_pool, idx_test = next(sss_test.split(X, y))
    X_pool, y_pool = X[idx_pool], y[idx_pool]
    X_test, y_test = X[idx_test], y[idx_test]

    sss_lab = StratifiedShuffleSplit(
        n_splits=1, test_size=(1 - labeled_frac), random_state=seed)
    idx_lab, idx_unlab = next(sss_lab.split(X_pool, y_pool))
    X_lab,   y_lab   = X_pool[idx_lab],   y_pool[idx_lab]
    X_unlab, y_unlab = X_pool[idx_unlab], y_pool[idx_unlab]

    log.info(f"  seed={seed} | labeled={len(X_lab):,} | "
             f"unlabeled={len(X_unlab):,} | test={len(X_test):,}")
    return X_lab, y_lab, X_unlab, y_unlab, X_test, y_test



### 6. Métricas de Evaluación

Esta función calcula un conjunto de métricas estándar para evaluar el rendimiento de los modelos de clasificación:

*   **F1-Score (Macro):** Promedio del F1-Score para ambas clases, útil para clases desbalanceadas.
*   **AUC (Area Under ROC Curve):** Mide la capacidad del modelo para distinguir entre clases.
*   **F1, Precision y Recall para la Clase 1:** Métricas específicas para la clase positiva ("alto consumo"), que suele ser la de mayor interés.

In [6]:
def eval_metrics(y_true, y_pred, y_prob) -> dict:
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro",  zero_division=0),
        "auc":      roc_auc_score(y_true, y_prob),
        "f1_0":     f1_score(y_true, y_pred, pos_label=0,      zero_division=0),
        "f1_1":     f1_score(y_true, y_pred, pos_label=1,      zero_division=0),
        "prec_1":   precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "rec_1":    recall_score(y_true,    y_pred, pos_label=1, zero_division=0),
    }

### 7. Modelos Baseline (Supervisados)

Aquí entrenamos dos modelos de referencia utilizando **únicamente el 10% de datos etiquetados**:

1.  **Regresión Logística (LR):** Un modelo lineal simple y robusto.
2.  **Random Forest (RF):** Un modelo de ensamble no lineal, más potente.

Es importante destacar que el Random Forest se calibra usando `CalibratedClassifierCV` con regresión isotónica. Esto ajusta las probabilidades predichas por el modelo para que sean más realistas y confiables, lo cual es un requisito fundamental para que el algoritmo de Self-Training funcione correctamente.

In [7]:
def run_baselines(X_lab, y_lab, X_test, y_test, seed: int):
    """
    LR y RF entrenados SOLO con el 10% etiquetado.
    El RF se calibra con isotonic regression para que sus probabilidades
    sean confiables al usarlas como umbral en SSL.
    """
    sc = StandardScaler().fit(X_lab)
    Xl = sc.transform(X_lab)
    Xt = sc.transform(X_test)

    # Logistic Regression — baseline lineal
    lr = LogisticRegression(
        class_weight="balanced", max_iter=2000,
        solver="lbfgs", random_state=seed)
    lr.fit(Xl, y_lab)
    res_lr = eval_metrics(y_test, lr.predict(Xt), lr.predict_proba(Xt)[:, 1])
    log.info(f"  LR  → F1={res_lr['f1_macro']:.3f}  "
             f"AUC={res_lr['auc']:.3f}  F1-cls1={res_lr['f1_1']:.3f}")

    # Random Forest calibrado — baseline no lineal
    base_rf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_features="sqrt", random_state=seed, n_jobs=-1)
    rf = CalibratedClassifierCV(base_rf, cv=3, method="isotonic")
    rf.fit(Xl, y_lab)
    res_rf = eval_metrics(y_test, rf.predict(Xt), rf.predict_proba(Xt)[:, 1])
    log.info(f"  RF  → F1={res_rf['f1_macro']:.3f}  "
             f"AUC={res_rf['auc']:.3f}  F1-cls1={res_rf['f1_1']:.3f}")

    return {"LR": res_lr, "RF": res_rf}, sc



### 8. Algoritmo de Self-Training (Semi-Supervised)

Esta es la implementación del clasificador Self-Training. El proceso es el siguiente:

1.  **Entrenar Modelo Inicial:** Se entrena un Random Forest (calibrado) solo con los datos etiquetados.
2.  **Predecir en Datos no Etiquetados:** El modelo predice las probabilidades de clase para todos los datos no etiquetados.
3.  **Generar Pseudo-Etiquetas:** Se seleccionan las predicciones que superan un umbral de confianza (`global_thresh`). Para evitar que el modelo se desvíe, se aplican varios controles:
    *   **Umbrales por Clase:** Se usa un umbral ligeramente más bajo para la clase minoritaria para ayudar a balancear las pseudo-etiquetas.
    *   **Límite de Adición:** En cada iteración, solo se añade un máximo del 20% del pool de datos no etiquetados restantes para evitar que una sola iteración domine el proceso.
    *   **Selección de los Mejores:** Solo se seleccionan las predicciones con la confianza más alta.
4.  **Re-entrenamiento:** Se añaden los datos con pseudo-etiquetas al conjunto de entrenamiento y se vuelve a entrenar el modelo.
5.  **Iterar:** Se repiten los pasos 2-4 durante un número definido de iteraciones (`n_iters`) o hasta que no se puedan añadir más pseudo-etiquetas.

Finalmente, el modelo entrenado con datos etiquetados y pseudo-etiquetados se evalúa en el conjunto de prueba.

In [8]:
def self_training_ssl(
    X_lab, y_lab, X_unlab, X_test, y_test, sc, seed: int,
    n_iters:       int   = 3,
    global_thresh: float = 0.75,   # umbral de confianza para pseudo-labels
    max_pool_frac: float = 0.20,   # requerimiento: máx 20% del pool/clase/iter
) -> dict:
    """
    Self-Training con todos los controles requeridos:
      ✓ Probabilidades calibradas (no overconfident)
      ✓ Per-class thresholds: clase minoritaria recibe umbral 5pp menor
      ✓ Pseudo-labels: solo los de MAYOR confianza por clase
      ✓ Límite 20% del pool restante por clase por iteración
      ✓ Test labels NUNCA usados durante SSL

    global_thresh = 0.75:
      Con probabilidades bien calibradas en un problema real,
      0.75 es conservador (el modelo necesita estar bastante seguro).
      0.90 sería demasiado restrictivo y añadiría 0 pseudo-labels.
    """
    Xl_cur    = sc.transform(X_lab).copy()
    Xu_all    = sc.transform(X_unlab).copy()
    yl_cur    = y_lab.copy()
    remaining = np.arange(len(Xu_all))

    # Per-class thresholds
    counts    = np.bincount(yl_cur)
    minor_cls = int(np.argmin(counts))
    thr = {0: global_thresh, 1: global_thresh}
    thr[minor_cls] -= 0.05
    log.info(f"    Thresholds → cls0={thr[0]:.2f}  cls1={thr[1]:.2f}  "
             f"(clase minoritaria: {minor_cls})")

    base = RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_features="sqrt", random_state=seed, n_jobs=-1)
    clf = CalibratedClassifierCV(base, cv=3, method="isotonic")

    for it in range(1, n_iters + 1):
        if len(remaining) == 0:
            log.info(f"    Iter {it}: pool agotado.")
            break

        Xu_rem = Xu_all[remaining]
        clf.fit(Xl_cur, yl_cur)
        proba  = clf.predict_proba(Xu_rem)          # (N_rem, 2)
        preds  = np.argmax(proba, axis=1)

        pseudo  = np.full(len(remaining), -1, dtype=int)
        max_add = max(1, int(max_pool_frac * len(remaining)))

        for cls in [0, 1]:
            conf           = proba[:, cls]
            candidate_mask = (preds == cls) & (conf >= thr[cls])
            candidate_idx  = np.where(candidate_mask)[0]
            if len(candidate_idx) == 0:
                continue
            # Ordenar por mayor confianza → seleccionar top max_add
            top = candidate_idx[np.argsort(conf[candidate_idx])[::-1]][:max_add]
            pseudo[top] = cls

        added = pseudo >= 0
        n_add = int(added.sum())

        if n_add == 0:
            log.info(f"    Iter {it}: 0 pseudo-labels (umbral no alcanzado). Stop.")
            break

        cls_cnt   = np.bincount(pseudo[added], minlength=2)
        Xl_cur    = np.vstack([Xl_cur, Xu_rem[added]])
        yl_cur    = np.concatenate([yl_cur, pseudo[added]])
        remaining = remaining[~added]

        log.info(f"    Iter {it}: +{n_add} pseudo-labels  "
                 f"(cls0={cls_cnt[0]}, cls1={cls_cnt[1]})  "
                 f"| train={len(yl_cur):,}  | pool={len(remaining):,}")

    # Entrenamiento final con labeled + pseudo-labels
    clf.fit(Xl_cur, yl_cur)
    Xt     = sc.transform(X_test)
    y_pred = clf.predict(Xt)
    y_prob = clf.predict_proba(Xt)[:, 1]

    log.info("\n" + classification_report(
        y_test, y_pred, target_names=["bajo", "alto"]))
    return eval_metrics(y_test, y_pred, y_prob)


### 9. Ejecución de Experimentos

Para asegurar que nuestros resultados son robustos y no dependen de una única división de datos aleatoria, ejecutamos todo el proceso (división, entrenamiento de baselines y SSL) varias veces con diferentes semillas (`SEEDS`).

La función `run_experiments` orquesta este proceso, almacenando los resultados de cada modelo en cada ejecución.

In [9]:
def run_experiments(X, y) -> dict:
    all_results = {"LR": [], "RF": [], "SSL_RF": []}

    for seed in SEEDS:
        log.info(f"\n{'='*60}\n  SEED {seed}\n{'='*60}")
        X_lab, y_lab, X_unlab, y_unlab, X_test, y_test = \
            get_splits(X, y, seed=seed)

        base_res, sc = run_baselines(X_lab, y_lab, X_test, y_test, seed)
        all_results["LR"].append(base_res["LR"])
        all_results["RF"].append(base_res["RF"])

        log.info(f"\n  [SSL Self-Training — seed {seed}]")
        ssl_res = self_training_ssl(
            X_lab, y_lab, X_unlab, X_test, y_test, sc, seed=seed,
            n_iters=3, global_thresh=0.75, max_pool_frac=0.20)
        all_results["SSL_RF"].append(ssl_res)
        log.info(f"  SSL → F1={ssl_res['f1_macro']:.3f}  "
                 f"AUC={ssl_res['auc']:.3f}")

    return all_results


### 10. Resumen y Almacenamiento de Resultados

Una vez finalizadas todas las ejecuciones, estas funciones se encargan de:

1.  **Calcular y Mostrar un Resumen:** La función `print_summary` calcula la media y la desviación estándar de las métricas para cada modelo a través de las diferentes semillas y las muestra en un formato de tabla claro.
2.  **Guardar Resultados:** La función `save_results` guarda dos archivos CSV:
    *   `challenge2_detail.csv`: Contiene los resultados detallados de cada modelo en cada ejecución (semilla).
    *   `challenge2_summary.csv`: Contiene la tabla resumen con las métricas agregadas.

In [10]:
def print_summary(all_results: dict) -> list:
    W = 72
    log.info(f"\n{'='*W}")
    log.info("  RESULTADOS FINALES — MEAN ± STD  (3 seeds)")
    log.info(f"{'='*W}")
    log.info(f"{'Modelo':<12}" + "".join(f"{k:>16}" for k in METRICS_KEYS))
    log.info(f"{'-'*W}")

    summary_rows = []
    for model, runs in all_results.items():
        row      = f"{model:<12}"
        row_data = {"model": model}
        for k in METRICS_KEYS:
            vals = np.array([r[k] for r in runs])
            m, s = vals.mean(), vals.std()
            row += f"  {m:.3f}±{s:.3f}"
            row_data[f"{k}_mean"] = round(m, 4)
            row_data[f"{k}_std"]  = round(s, 4)
        log.info(row)
        summary_rows.append(row_data)
    log.info(f"{'='*W}")
    return summary_rows


def save_results(all_results: dict, summary_rows: list):
    detail = []
    for model, runs in all_results.items():
        for s, r in zip(SEEDS, runs):
            detail.append({"model": model, "seed": s, **r})

    pd.DataFrame(detail).to_csv("results/challenge2_detail.csv",  index=False)
    pd.DataFrame(summary_rows).to_csv("results/challenge2_summary.csv", index=False)
    log.info("✓ results/challenge2_detail.csv")
    log.info("✓ results/challenge2_summary.csv")
    log.info("✓ logs/challenge2.log")

### 11. Orquestación Principal

Este es el punto de entrada del script. El bloque `if __name__ == "__main__":` asegura que el código se ejecute solo cuando el script es llamado directamente. Orquesta la ejecución de todas las funciones en el orden correcto:

1.  Descarga los datos.
2.  Preprocesa los datos para obtener `X` y `y`.
3.  Ejecuta los experimentos con múltiples semillas.
4.  Imprime el resumen de resultados.
5.  Guarda los resultados en archivos CSV.

In [12]:
if __name__ == "__main__":
    raw_df      = fetch_eia_retail_sales(API_KEY)
    X, y        = preprocess(raw_df)
    all_results = run_experiments(X, y)
    summary     = print_summary(all_results)
    save_results(all_results, summary)
 

2026-04-09 22:03:10,939 | INFO | Cargando datos desde caché: eia_retail_sales.csv
2026-04-09 22:03:11,217 | INFO |   111,972 registros cargados
2026-04-09 22:03:11,365 | INFO | Registros iniciales: 93,310 | Estados: 62 | Rango: 2001-01-01 → 2026-01-01
2026-04-09 22:03:11,501 | INFO | Target (top 30% dentro del estado): clase 1 = 33.4% | clase 0 = 66.6%
2026-04-09 22:03:11,714 | INFO | Dataset final: 67,270 filas × 19 features
2026-04-09 22:03:11,714 | INFO | Clases: 0 = 45,038 | 1 = 22,232
2026-04-09 22:03:11,731 | INFO | 
  SEED 42
2026-04-09 22:03:11,798 | INFO |   seed=42 | labeled=5,381 | unlabeled=48,435 | test=13,454
2026-04-09 22:03:11,875 | INFO |   LR  → F1=0.793  AUC=0.876  F1-cls1=0.736
2026-04-09 22:03:16,094 | INFO |   RF  → F1=0.922  AUC=0.982  F1-cls1=0.894
2026-04-09 22:03:16,094 | INFO | 
  [SSL Self-Training — seed 42]
2026-04-09 22:03:16,110 | INFO |     Thresholds → cls0=0.75  cls1=0.70  (clase minoritaria: 1)
2026-04-09 22:03:20,389 | INFO |     Iter 1: +19374 pseu